# Resultados de clasificación

Primero la métrica del **bucle externo** (media ± desviación de 3 pliegues) para las 112 combinaciones; después la evaluación única en la **prueba 2023–2024** de los reajustes finales, con el umbral elegido en 2022.

In [1]:
import os  # Rutas
import sys  # Configuración del intérprete
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count()))  # Evita un aviso de joblib en Windows
sys.path[:0] = [os.path.abspath(".."), os.path.abspath("../.."), os.path.abspath("../../entregable1")]  # e3, src y e1
import warnings  # Control de avisos
warnings.filterwarnings("ignore")  # Oculta avisos de versiones
import contextlib, io  # Silenciar un diagnóstico de joblib en Windows
from joblib.externals.loky import cpu_count  # Conteo de núcleos
with contextlib.redirect_stderr(io.StringIO()):  # joblib imprime un diagnóstico al no encontrar «wmic»
    cpu_count(only_physical_cores=True)  # Se calcula una vez (queda en caché)
import pickle  # Lectura de resultados
import numpy as np  # Cálculo numérico
import pandas as pd  # Tablas
import plotly.express as px  # Figuras rápidas
import plotly.graph_objects as go  # Figuras detalladas
from plotly.subplots import make_subplots  # Paneles
from IPython.display import Markdown, display  # Texto enriquecido
from e1.figuras import mostrar, AZUL, ROJO, VERDE, AMBAR, GRIS, MORADO  # Estilo de figuras (tema claro del informe)
from e3 import config as c3, datos as dd, modelos as mo, optimizadores as op, experimento as ex  # Código del entregable
c3.fijar_semilla()  # Semilla global (42)
pd.set_option("display.width", 170)  # Ancho de las tablas
pd.set_option("display.max_columns", 30)  # Columnas visibles
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")  # Cuatro decimales
COLORES = {"Grid Search": GRIS, "Random Search": AMBAR, "Bayesiana (Optuna TPE)": AZUL, "Genética (DEAP)": VERDE, "Successive Halving": MORADO}  # Color por optimizador
from e3 import evaluacion as ev  # Métricas y calibración
from sklearn.metrics import roc_curve  # Curva ROC
tabla = pd.read_csv(c3.RESULTADOS / "tabla_maestra.csv")  # Tabla maestra
clf = tabla[tabla["tarea"] == "clf"]  # Clasificación
externo = clf.groupby(["modelo", "balanceo", "optimizador"]).agg(**{"AUC-PR media": ("AUC-PR", "mean"), "AUC-PR desv.": ("AUC-PR", "std"),
                                                                  "AUC-ROC media": ("AUC-ROC", "mean"), "AUC-ROC desv.": ("AUC-ROC", "std"),
                                                                  "Brier media": ("Brier", "mean"), "segundos": ("segundos_optimizacion", "sum")}).sort_values("AUC-PR media", ascending=False)
print(f"Combinaciones de clasificación: {len(externo)}")  # 112
externo.head(15)

Combinaciones de clasificación: 112


AUC-PR media  AUC-PR desv.  AUC-ROC media  AUC-ROC desv.  Brier media   segundos
modelo        balanceo              optimizador                                                                                             
XGBoost       Sin balanceo          Genética (DEAP)               0.6792        0.1041         0.9168         0.0122       0.0675   192.6876
              class_weight balanced Random Search                 0.6781        0.1041         0.9182         0.0125       0.1109   171.8169
                                    Genética (DEAP)               0.6779        0.1038         0.9171         0.0133       0.1047   194.2503
Random Forest class_weight balanced Genética (DEAP)               0.6778        0.1068         0.9163         0.0115       0.1050   913.8178
              Sin balanceo          Bayesiana (Optuna TPE)        0.6777        0.1070         0.9148         0.0107       0.0667 1,384.7373
XGBoost       Sin balanceo          Bayesiana (Optuna TPE)        0.6772        0.1055         0.9173         0.0128       0.0666   126.0296
                                    Random Search                 0.6768        0.1053         0.9161         0.0120       0.0714   144.8126
Random Forest class_weight balanced Random Search                 0.6767        0.1045         0.9165         0.0118       0.1078 1,215.5588
              Sin balanceo          Genética (DEAP)               0.6763        0.1092         0.9154         0.0104       0.0668   936.5875
                                    Grid Search                   0.6759        0.1103         0.9143         0.0089       0.0668   984.0217
                                    Random Search                 0.6758        0.1091         0.9145         0.0106       0.0666 1,330.8958
XGBoost       Sin balanceo          Grid Search                   0.6742        0.1052         0.9164         0.0127       0.0689   112.7373
              class_weight balanced Bayesiana (Optuna TPE)        0.6740        0.1047         0.9170         0.0120       0.1114   126.2842
                                    Grid Search                   0.6730        0.1064         0.9161         0.0118       0.0915   147.9220
Random Forest class_weight balanced Grid Search                   0.6727        0.1062         0.9158         0.0122       0.0998   921.1852

In [2]:
mejores = externo.reset_index().groupby("modelo").first().sort_values("AUC-PR media", ascending=False)  # Mejor combinación de cada modelo (según el bucle externo)
mejores[["balanceo", "optimizador", "AUC-PR media", "AUC-PR desv.", "AUC-ROC media", "AUC-ROC desv."]]

,balanceo,optimizador,AUC-PR media,AUC-PR desv.,AUC-ROC media,AUC-ROC desv.
modelo,,,,,,
XGBoost,Sin balanceo,Genética (DEAP),0.6792,0.1041,0.9168,0.0122
Random Forest,class_weight balanced,Genética (DEAP),0.6778,0.1068,0.9163,0.0115
SVM,Sin balanceo,Genética (DEAP),0.6706,0.1068,0.9138,0.0117
Regresión logística,Sin balanceo,Genética (DEAP),0.6532,0.1120,0.9007,0.0051
KNN,Sin balanceo,Random Search,0.6506,0.1160,0.8969,0.0046
Árbol de decisión,class_weight balanced,Bayesiana (Optuna TPE),0.6340,0.1146,0.8954,0.0047
Naive Bayes,ADASYN,Bayesiana (Optuna TPE),0.5460,0.1374,0.8874,0.0035


**Interpretación.** En el bucle externo el mejor es **XGBoost sin balanceo** optimizado con el algoritmo genético (AUC-PR 0,679 ± 0,104), seguido muy de cerca por Random Forest (0,678) y la SVM con núcleo aproximado (0,671). Los modelos lineales y de vecindad quedan unas 2–3 centésimas atrás (regresión logística 0,653, KNN 0,651), el árbol único en 0,634 y Naive Bayes bastante más lejos (0,546): su supuesto de independencia condicional choca con los bloques de variables colineales del EDA. La **desviación entre pliegues externos (≈ 0,10) es mucho mayor que las diferencias entre los mejores modelos (≈ 0,001–0,01)**: cambia más el año evaluado (2019 epidémico frente a 2021 de baja transmisión) que el modelo. Por eso las diferencias se juzgan con pruebas estadísticas formales (capítulo 8) y no a simple vista.

## Efecto del balanceo

In [3]:
bal = clf.pivot_table(index=["modelo", "optimizador", "anio_externo"], columns="balanceo", values="AUC-PR")  # Pareado: mismo modelo, optimizador y pliegue
dif = bal.sub(bal["Sin balanceo"], axis=0).drop(columns="Sin balanceo")  # Diferencia frente a no balancear
fig = go.Figure()  # Diferencias por modelo
for b, col in zip(dif.columns, [AMBAR, ROJO, AZUL]):  # Estrategias
    g = dif[b].groupby(level="modelo").mean()  # Media por modelo
    fig.add_bar(x=g.index, y=g.values, name=b, marker_color=col)  # Barras
fig.add_hline(y=0, line_color=GRIS)  # Sin efecto
fig.update_layout(barmode="group", title="AUC-PR externo de cada estrategia de balanceo menos «Sin balanceo» (media por modelo)", yaxis_title="Δ AUC-PR")  # Formato
mostrar(fig, 400)  # Muestra
pd.DataFrame({"Δ AUC-PR medio": dif.mean(), "veces que mejora": (dif > 0).mean(), "Δ AUC-ROC medio":
              clf.pivot_table(index=["modelo", "optimizador", "anio_externo"], columns="balanceo", values="AUC-ROC").pipe(lambda t: t.sub(t["Sin balanceo"], axis=0)).drop(columns="Sin balanceo").mean()})

,Δ AUC-PR medio,veces que mejora,Δ AUC-ROC medio
balanceo,,,
ADASYN,-0.0241,0.1667,-0.0036
SMOTE,-0.0101,0.1310,-0.0016
class_weight balanced,-0.0053,0.2024,-0.0009


**Interpretación.** El balanceo **no mejora** la discriminación: en promedio, ADASYN reduce el AUC-PR externo en 0,024, SMOTE en 0,010 y `class_weight` en 0,005, y solo mejora en el 13–20 % de los casos pareados. El AUC-ROC casi no cambia. Hay tres razones: (1) el AUC-ROC y el AUC-PR no dependen del umbral, y pesar o replicar la clase minoritaria cambia la escala de las probabilidades más que su orden; (2) en un panel temporal, los brotes sintéticos interpolan semanas de municipios y años distintos y agregan ruido, sobre todo ADASYN, que los concentra en la frontera más confusa; y (3) con ≈ 10 % de brotes y más de 13 000 casos de la clase minoritaria, el desbalance no es extremo. El desbalance se maneja mejor **eligiendo el umbral** en 2022: así se gana recall sin deformar las probabilidades.

## Evaluación en la prueba 2023–2024

Cada combinación se reajustó con 2015–2021; su umbral maximiza F1 en 2022. La SVM entrega márgenes: para sus métricas de probabilidad (Brier) se usa el escalado de Platt ajustado en 2022.

In [4]:
tr, va, te = dd.cargar()  # Periodos
y_va, y_te = va["y"].to_numpy(), te["y"].to_numpy()  # Objetivos
finales = {}  # Predicciones de las 112 combinaciones
for f in sorted(ex.FINALES.glob("clf__*.pkl")):  # Cada reajuste final
    r = pickle.load(open(f, "rb"))  # Predicciones
    finales[(r["modelo"], r["balanceo"], r["optimizador"])] = r  # Indexado
filas = []  # Métricas en la prueba
for k, r in finales.items():  # Cada combinación
    p_va, p_te = r["p_va"].astype(float), r["p_te"].astype(float)  # Puntajes
    if k[0] == "SVM":  # Márgenes → probabilidad con Platt (ajustado en 2022)
        p_te, _ = ev.recalibrar(p_va, y_va, p_te)  # Platt
        p_va, _ = ev.recalibrar(p_va, y_va, p_va)  # Validación en la misma escala
    u = ev.umbral_f1(y_va, p_va)  # Umbral de 2022
    filas.append({"modelo": k[0], "balanceo": k[1], "optimizador": k[2], "umbral": u, **ev.metricas_clf(y_te, p_te, u), "ECE": ev.ece(y_te, p_te)})  # Métricas
prueba = pd.DataFrame(filas).set_index(["modelo", "balanceo", "optimizador"])  # Tabla
sel = [tuple(x) for x in mejores.reset_index()[["modelo", "balanceo", "optimizador"]].to_numpy()]  # Las 7 elegidas por el bucle externo
prueba.loc[sel, ["Accuracy", "Precision", "Recall", "F1", "AUC-ROC", "AUC-PR", "Brier", "ECE", "umbral"]].sort_values("AUC-PR", ascending=False)

,,,Accuracy,Precision,Recall,F1,AUC-ROC,AUC-PR,Brier,ECE,umbral
modelo,balanceo,optimizador,,,,,,,,,
XGBoost,Sin balanceo,Genética (DEAP),0.8300,0.7045,0.7562,0.7295,0.8977,0.8225,0.1149,0.0383,0.3048
SVM,Sin balanceo,Genética (DEAP),0.8089,0.6439,0.8263,0.7238,0.8967,0.8204,0.1178,0.0467,0.1778
Random Forest,class_weight balanced,Genética (DEAP),0.8275,0.6946,0.7691,0.7299,0.8963,0.8184,0.1738,0.2055,0.7476
Regresión logística,Sin balanceo,Genética (DEAP),0.8206,0.6832,0.7609,0.7200,0.8856,0.8076,0.1217,0.0445,0.2215
KNN,Sin balanceo,Random Search,0.8154,0.6648,0.7886,0.7214,0.8853,0.8008,0.1239,0.0653,0.2046
Árbol de decisión,class_weight balanced,Bayesiana (Optuna TPE),0.8192,0.6761,0.7744,0.7219,0.8845,0.7920,0.1911,0.2138,0.7798
Naive Bayes,ADASYN,Bayesiana (Optuna TPE),0.8110,0.6664,0.7534,0.7072,0.8621,0.6757,0.2102,0.1922,0.9999


**Interpretación.** En la prueba 2023–2024, **XGBoost sin balanceo** es el mejor en AUC-PR (0,823) y AUC-ROC (0,898), con F1 0,73: anticipa el 76 % de los brotes con 70 % de precisión. Supera al modelo base logístico de los entregables anteriores (AUC-PR 0,809). La SVM alcanza casi el mismo AUC-PR (0,820) pero con un umbral más bajo: más recall (0,83) a costa de precisión (0,64). El orden de los modelos en la prueba coincide en lo esencial con el del bucle externo, lo que indica que la validación anidada estimó bien la generalización. Naive Bayes vuelve a ser el peor (AUC-PR 0,676).

In [5]:
prueba.sort_values("AUC-PR", ascending=False)  # Las 112 combinaciones en la prueba

umbral  Accuracy  Precision  Recall     F1  AUC-ROC  AUC-PR  Brier    ECE
modelo      balanceo              optimizador                                                                                      
XGBoost     class_weight balanced Genética (DEAP)         0.6807    0.8093     0.6429  0.8339 0.7260   0.8981  0.8226 0.1784 0.2161
            Sin balanceo          Genética (DEAP)         0.3048    0.8300     0.7045  0.7562 0.7295   0.8977  0.8225 0.1149 0.0383
                                  Grid Search             0.2199    0.8090     0.6436  0.8287 0.7245   0.8974  0.8224 0.1144 0.0342
            class_weight balanced Bayesiana (Optuna TPE)  0.7526    0.8270     0.6917  0.7745 0.7308   0.8976  0.8221 0.1771 0.2107
                                  Random Search           0.6624    0.7953     0.6169  0.8562 0.7171   0.8981  0.8221 0.1833 0.2226
...                                                          ...       ...        ...     ...    ...      ...     ...    ...    ...
Naive Bayes class_weight balanced Random Search           0.9957    0.8067     0.6556  0.7627 0.7051   0.8592  0.6733 0.2014 0.1988
                                  Grid Search             0.9957    0.8067     0.6556  0.7627 0.7051   0.8592  0.6733 0.2014 0.1988
                                  Genética (DEAP)         0.9957    0.8067     0.6556  0.7627 0.7051   0.8592  0.6733 0.2014 0.1988
            ADASYN                Genética (DEAP)         0.9999    0.8113     0.6673  0.7523 0.7073   0.8614  0.6732 0.2108 0.1931
            SMOTE                 Genética (DEAP)         0.3675    0.7891     0.6155  0.8105 0.6997   0.8586  0.6731 0.2013 0.1988

[112 rows x 9 columns]

## Matrices de confusión y curvas ROC de los 7 modelos

In [6]:
fig = make_subplots(2, 4, subplot_titles=[f"{m}<br><sup>{b} · {o}</sup>" for m, b, o in sel] + ["Curvas ROC"], specs=[[{}] * 4, [{}] * 3 + [{}]])  # Rejilla
for j, k in enumerate(sel):  # Cada modelo
    r = finales[k]  # Predicciones
    p_va, p_te = r["p_va"].astype(float), r["p_te"].astype(float)  # Puntajes
    u = ev.umbral_f1(y_va, p_va)  # Umbral
    cm = ev.matriz(y_te, p_te, u)  # Matriz
    fila, col = (j // 4) + 1, (j % 4) + 1  # Posición
    if j < 7:  # Matrices
        fig.add_heatmap(z=cm, x=["pred. no", "pred. sí"], y=["real no", "real sí"], text=[[f"VN {cm[0, 0]:,}", f"FP {cm[0, 1]:,}"], [f"FN {cm[1, 0]:,}", f"VP {cm[1, 1]:,}"]],
                        texttemplate="%{text}", colorscale="Blues", showscale=False, row=fila, col=col)
        fig.update_yaxes(autorange="reversed", row=fila, col=col)  # Real arriba
roc_fila, roc_col = 2, 4  # Panel de las curvas ROC
for j, k in enumerate(sel):  # Curvas
    fpr, tpr, _ = roc_curve(y_te, finales[k]["p_te"].astype(float))  # ROC
    fig.add_scatter(x=fpr, y=tpr, name=k[0], line=dict(color=px.colors.qualitative.Bold[j]), row=roc_fila, col=roc_col)  # Curva
fig.add_scatter(x=[0, 1], y=[0, 1], line=dict(color=GRIS, dash="dot"), showlegend=False, row=roc_fila, col=roc_col)  # Azar
fig.update_layout(title="Prueba 2023–2024: matriz de confusión (umbral de 2022) y curva ROC del mejor de cada modelo")  # Título
mostrar(fig, 680)  # Muestra

**Interpretación.** Las matrices muestran el compromiso de cada umbral elegido en 2022: los modelos con umbral bajo (SVM, KNN) tienen más verdaderos positivos y más falsas alarmas; los de umbral alto, al revés. Las curvas ROC de XGBoost, Random Forest y SVM prácticamente se superponen; la de Naive Bayes queda claramente por debajo.

## Calibración de probabilidades

Diagrama de confiabilidad, Brier y ECE; después, recalibración con **Platt** (logística sobre el logit) y **regresión isotónica**, ambas ajustadas en 2022 y evaluadas en la prueba.

In [7]:
fig = make_subplots(1, 2, subplot_titles=("Sin recalibrar", "Recalibrado con Platt (2022)"))  # Dos paneles
filas = []  # Efecto de la recalibración
for j, k in enumerate(sel):  # Cada modelo
    p_va, p_te = finales[k]["p_va"].astype(float), finales[k]["p_te"].astype(float)  # Puntajes
    platt, iso = ev.recalibrar(p_va, y_va, p_te)  # Recalibradas
    crudo = p_te if k[0] != "SVM" else platt  # La SVM no tiene probabilidad cruda
    col = px.colors.qualitative.Bold[j]  # Color
    for c, p in [(1, crudo), (2, platt)]:  # Paneles
        cf_ = ev.confiabilidad(y_te, p)  # Diagrama
        fig.add_scatter(x=cf_["predicha"], y=cf_["observada"], mode="lines+markers", name=k[0], line=dict(color=col), showlegend=c == 1, row=1, col=c)  # Curva
    filas.append({"modelo": k[0], "balanceo": k[1], "Brier crudo": ev.metricas_clf(y_te, crudo, 0.5).get("Brier", np.nan), "ECE crudo": ev.ece(y_te, crudo),
                  "Brier Platt": ev.metricas_clf(y_te, platt, 0.5)["Brier"], "ECE Platt": ev.ece(y_te, platt),
                  "Brier isotónica": ev.metricas_clf(y_te, iso, 0.5)["Brier"], "ECE isotónica": ev.ece(y_te, iso)})
for c in (1, 2):  # Diagonal
    fig.add_scatter(x=[0, 1], y=[0, 1], line=dict(color=GRIS, dash="dot"), showlegend=False, row=1, col=c)
fig.update_xaxes(title_text="probabilidad predicha")  # Ejes
fig.update_yaxes(title_text="frecuencia observada")  # Ejes
mostrar(fig, 440)  # Muestra
pd.DataFrame(filas).set_index("modelo")

,balanceo,Brier crudo,ECE crudo,Brier Platt,ECE Platt,Brier isotónica,ECE isotónica
modelo,,,,,,,
XGBoost,Sin balanceo,0.1149,0.0383,0.1131,0.0155,0.1138,0.0173
Random Forest,class_weight balanced,0.1738,0.2055,0.1135,0.0086,0.1143,0.0159
SVM,Sin balanceo,0.1178,0.0467,0.1178,0.0467,0.1139,0.0124
Regresión logística,Sin balanceo,0.1217,0.0445,0.1206,0.0374,0.1188,0.0304
KNN,Sin balanceo,0.1239,0.0653,0.1193,0.0160,0.1193,0.0140
Árbol de decisión,class_weight balanced,0.1911,0.2138,0.1194,0.0329,0.1179,0.0153
Naive Bayes,ADASYN,0.2102,0.1922,0.1388,0.0584,0.1345,0.0370


In [8]:
cal_bal = prueba.groupby("balanceo")[["Brier", "ECE", "AUC-ROC", "Recall", "Precision"]].mean()  # Efecto del balanceo en la calibración (promedio de modelos y optimizadores)
cal_bal

,Brier,ECE,AUC-ROC,Recall,Precision
balanceo,,,,,
ADASYN,0.1722,0.1627,0.8805,0.7941,0.6458
SMOTE,0.1580,0.1445,0.8840,0.7850,0.6569
Sin balanceo,0.1305,0.0668,0.8860,0.7785,0.6707
class_weight balanced,0.1722,0.1783,0.8842,0.7869,0.6615


**Interpretación.** Los modelos **sin balanceo** están razonablemente calibrados (ECE ≈ 0,04–0,07); los que usaron balanceo (Random Forest y árbol con `class_weight`, Naive Bayes con ADASYN) tienen ECE ≈ 0,19–0,21: aprendieron una prevalencia ficticia del 50 % y **sobreestiman** el riesgo (sus umbrales óptimos suben a 0,75–1,0). Recalibrar con **Platt** en 2022 baja el ECE a 0,01–0,06 y el Brier de Random Forest de 0,174 a 0,114; la **regresión isotónica** logra un efecto parecido y es mejor para la SVM (0,047 → 0,012). En promedio sobre las 112 combinaciones, no balancear deja el mejor Brier (0,131) y ECE (0,067); el balanceo sube el recall solo ≈ 1 punto. Conclusión: el balanceo **cambia la calibración, no la discriminación**.